# Importa Tudo

### Bibliotecas

In [ ]:
# Incorporar métricas de aluguel estimado (usar os dados do edsão?) alem de custos de venda, corretagem
# Gerar KPIs, como ROI, TIR, comparar ao CDI e etc... (fazer a mesma comparação para outros investimentos) 
# Calcular tempo payback
# Desvio padrão pra analisar os riscos (value at risk?)
# Gráficos de séries temporais
# Comparativo com outros investimentos
# Todos os calculos de custos tributários devem estar envolvidos

# Mapa interativo com POIs (fazer com qGis ou felt)
# Esse mapa pode ter todos os principais POIs
# Densidade mapa por tipo de serviço (saude, lazer, educacao, transporte)
# Acesso a parques, áreas verdes
# Todos os indicadores sociais
# Preco do metro quadrado médio da regiao

# Descritivo bairro do chatgpt
# Descrição Geral do Bairro:
# Use o ChatGPT para gerar um descritivo detalhado do bairro, abordando:
# História: breve histórico da região.
# Características principais: aspectos de segurança, perfil dos moradores (jovens, famílias, aposentados), cultura e estilo de vida.
# Desenvolvimento futuro: perspectiva de desenvolvimento do bairro, como novos empreendimentos, melhorias em infraestrutura, etc.
# Compare o bairro com outros bairros semelhantes da cidade em termos de preço médio por metro quadrado, qualidade de vida, infraestrutura, etc.

# Dados do imovel
# planta baixa gerada por IA
# mapeamento VR feito por matterport

### Ver se essas classes estao validas

In [1]:
import pandas as pd
from functools import reduce
import yfinance as yf

class DadosEconomicos:
    def __init__(self):
        self.month_mapping = {'JAN': '01', 'FEV': '02', 'MAR': '03', 'ABR': '04', 'MAI': '05', 'JUN': '06',
                              'JUL': '07', 'AGO': '08', 'SET': '09', 'OUT': '10', 'NOV': '11', 'DEZ': '12'}
        self.month_names = {'Janeiro': '01', 'Fevereiro': '02', 'Março': '03', 'Abril': '04',
                            'Maio': '05', 'Junho': '06', 'Julho': '07', 'Agosto': '08', 'Setembro': '09',
                            'Outubro': '10', 'Novembro': '11', 'Dezembro': '12'}

    def formatacao_yahii(self, url, column_name, date_column='A/M'):
        df = pd.read_html(url, encoding='latin1')[2]
        df.columns = df.iloc[0]
        df = df.iloc[1:]
        df = df[pd.to_numeric(df[date_column], errors='coerce').notnull()]
        df = df.melt(id_vars=date_column, var_name='Month', value_name='Value')
        df['Data'] = df[date_column] + '-' + df['Month'].map(self.month_mapping)
        df['Data'] = pd.to_datetime(df['Data'], format='%Y-%m')
        df['Value'] = (df['Value']
                       .str.replace('%', '')
                       .str.replace('.', '')
                       .str.replace(',', '.')
                       .str.replace('(', '')
                       .str.replace(')', '')
                       .astype(float))
        df = df[['Data', 'Value']]
        df.columns = ['Data', column_name]
        df.dropna(inplace=True)
        return df

    def calculate_variation(self, df, column_name):
        df = df.sort_values('Data')
        df[column_name] = df[column_name].pct_change() * 100
        return df

    def scraping_dolar(self):
        url = 'http://www.yahii.com.br/dolar.html'
        try:
            dolar = pd.read_html(url, encoding='ISO-8859-1')[3]
        except UnicodeDecodeError:
            dolar = pd.read_html(url, encoding='cp1252')[3]
        dolar.columns = dolar.iloc[0]
        dolar = dolar.iloc[1:]
        dolar = dolar[pd.to_numeric(dolar['A/M'], errors='coerce').notnull()]
        dolar = dolar.melt(id_vars='A/M', var_name='Month', value_name='Value')
        dolar['Data'] = dolar['A/M'] + '-' + dolar['Month'].map(self.month_mapping)
        dolar['Data'] = pd.to_datetime(dolar['Data'], format='%Y-%m')
        dolar['Value'] = dolar['Value'].astype(float) / 10000
        dolar = dolar[['Data', 'Value']]
        dolar.columns = ['Data', 'Dolar']
        dolar.dropna(subset=['Dolar'], inplace=True)
        return self.calculate_variation(dolar, 'Dolar')

    def scraping_selic(self):
        url = 'https://www.gov.br/receitafederal/pt-br/assuntos/orientacao-tributaria/pagamentos-e-parcelamentos/taxa-de-juros-selic#Taxa_de_Juros_Selic'
        selic = pd.read_html(url)
        selic = pd.concat(objs=[selic[2], selic[1].iloc[:, 1:]], axis=1)
        selic.columns = selic.iloc[0]
        selic = selic[1:]
        selic2 = pd.read_html(url)
        selic2 = pd.concat(objs=[selic2[4], selic2[3]], axis=1)
        selic2.columns = selic2.iloc[0]
        selic2 = selic2[1:]
        selic2 = selic2.drop(columns=['Mês/Ano'])
        selic = pd.concat([selic, selic2], axis=1)
        selic = selic.melt(id_vars=['Mês/Ano'], var_name='Year', value_name='Value')
        selic['Data'] = selic['Mês/Ano'] + '-' + selic['Year'].astype(str)
        selic = selic.drop(columns=['Mês/Ano', 'Year'])
        selic = selic.rename(columns={'Value': 'Selic'})
        selic['Data'] = selic['Data'].apply(lambda x: f"{x.split('-')[1]}-{self.month_names[x.split('-')[0]]}-01")
        selic['Data'] = pd.to_datetime(selic['Data'])
        selic = selic[selic['Data'] > '2000-01-01']
        selic['Selic'] = selic['Selic'].str.replace(',', '.').str.replace('%', '').astype(float)
        return selic

    def scraping_ipca(self):
        url = 'https://www.mobills.com.br/tabelas/ipca/'
        ipca = pd.read_html(url)[3]
        ipca.columns = ipca.iloc[0]
        ipca = ipca[1:]
        ipca = pd.melt(ipca, id_vars=['IPCA histórico'], var_name='yearmonth', value_name='value')
        ipca = ipca[ipca['yearmonth'] != 'Acumulado ao ano %']
        ipca['Data'] = ipca['yearmonth'] + '-' + ipca['IPCA histórico'].astype(str)
        ipca = ipca.rename(columns={'value': 'Ipca'})
        ipca = ipca.drop(columns=['IPCA histórico', 'yearmonth'])
        ipca['Data'] = ipca['Data'].apply(lambda x: f"{x.split('-')[1]}-{self.month_names[x.split('-')[0]]}-01")
        ipca['Data'] = pd.to_datetime(ipca['Data'])
        ipca = ipca[ipca['Data'] > '2000-01-01']
        ipca['Ipca'] = pd.to_numeric(ipca['Ipca'].str.replace('%', '').str.replace(',', ''), errors='coerce') / 100
        return ipca

    def scraping_igpm(self):
        igpm1 = self.formatacao_yahii('http://www.yahii.com.br/igpm1989a2008.html', 'Igpm')
        igpm2 = self.formatacao_yahii('http://www.yahii.com.br/igpm.html', 'Igpm')
        igpm = pd.concat([igpm1, igpm2])
        return igpm

    def scraping_incc(self):
        incc = self.formatacao_yahii('http://www.yahii.com.br/inccM.html', 'Incc')
        return incc

    def scraping_ivgr(self):
        url = 'https://www3.bcb.gov.br/sgspub/consultarvalores/consultarValoresSeries.do?method=consultarSeries&series=21340'
        ivgr = pd.read_html(url, encoding='latin1')[4]
        ivgr.columns = ivgr.iloc[2]
        ivgr.drop([0, 1, 2], inplace=True)
        ivgr['Date month/YYYY'] = pd.to_datetime(ivgr['Date month/YYYY'], errors='coerce')
        ivgr.dropna(subset=['Date month/YYYY'], inplace=True)
        ivgr['21340  Index'] = ivgr['21340  Index'].astype(float) / 100
        ivgr.columns = ['Data', 'Ivgr']
        return self.calculate_variation(ivgr, 'Ivgr')

    def scraping_ipam(self):
        ipam = self.formatacao_yahii('http://www.yahii.com.br/ipaM.html', 'Ipam')
        return ipam

    def scraping_ipcfipe(self):
        ipcfipe1 = self.formatacao_yahii('http://www.yahii.com.br/ipcfipe90a09.html', 'Ipcfipe')
        ipcfipe2 = self.formatacao_yahii('http://www.yahii.com.br/ipcfipe.html', 'Ipcfipe')
        ipcfipe = pd.concat([ipcfipe1, ipcfipe2])
        return ipcfipe

    def scraping_iiebr(self):
        url = 'http://www.yahii.com.br/iieBR.html'
        iiebr_list = pd.read_html(url, encoding='latin1')
        dfs = []
        for i in range(2, 6):
            df = iiebr_list[i]
            df.columns = df.iloc[0]
            df = df.iloc[1:]
            df = df.melt(id_vars='M/A', var_name='Month', value_name='Value')
            df['M/A'] = df['M/A'].map(self.month_mapping)
            df['Data'] = df['M/A'].astype(int).astype(str) + '-' + df['Month'].astype(int).astype(str)
            df['Data'] = pd.to_datetime(df['Data'], format='%m-%Y')
            df['Value'] = df['Value'] / 10
            df = df[['Data', 'Value']]
            df.columns = ['Data', 'Iiebr']
            df.dropna(subset=['Iiebr'], inplace=True)
            dfs.append(df)
        iiebr = pd.concat(dfs, ignore_index=True)
        return self.calculate_variation(iiebr, 'Iiebr')

    def scraping_cubsp(self):
        def process_df(df):
            df.columns = df.iloc[0]
            df = df.iloc[1:]
            df['M/A'] = df['M/A'].map(self.month_mapping)
            df = df.melt(id_vars='M/A', var_name='Month', value_name='Value')
            df['Data'] = df['M/A'].astype(str) + '-' + df['Month'].astype(str)
            df['Data'] = pd.to_datetime(df['Data'], format='%m-%Y', errors='coerce')
            df['Value'] = (df['Value']
                           .str.replace('%', '')
                           .str.replace('.', '')
                           .str.replace(',', '.')
                           .str.replace('(', '')
                           .str.replace(')', '')
                           .astype(float))
            df = df[['Data', 'Value']]
            df.columns = ['Data', 'Cubsp']
            df.dropna(subset=['Cubsp'], inplace=True)
            return df

        urls = ['http://www.yahii.com.br/cubsp94a01.html', 'http://www.yahii.com.br/cubsp02a09.html',
                'http://www.yahii.com.br/cubsp10a17.html', 'http://www.yahii.com.br/cubsp.html']
        all_dfs = []
        for url in urls:
            df = pd.read_html(url, encoding='latin1')[2]
            processed_df = process_df(df)
            all_dfs.append(processed_df)
        cubsp = pd.concat(all_dfs, ignore_index=True)
        return cubsp

    def concatenar_tudo(self):
        igpm = self.scraping_igpm()
        selic = self.scraping_selic()
        ipca = self.scraping_ipca()
        dolar = self.scraping_dolar()
        incc = self.scraping_incc()
        ivgr = self.scraping_ivgr()
        ipam = self.scraping_ipam()
        ipcfipe = self.scraping_ipcfipe()
        iiebr = self.scraping_iiebr()
        cubsp = self.scraping_cubsp()

        dataframes = [igpm, selic, ipca, dolar, incc, ivgr, ipam, ipcfipe, iiebr, cubsp]
        economia = reduce(lambda left, right: pd.merge(left, right, on='Data', how='left'), dataframes)
        return economia


###########-------------------------------------------------------###########

def pega_ativos_yfinance(tickers):
    # Link tesouro direto (caso precise no futuro)
    # https://www.tesourotransparente.gov.br/ckan/dataset/df56aa42-484a-4a59-8184-7676580c81e3/resource/796d2059-14e9-44e3-80c9-2d9e30b405c1/download/PrecoTaxaTesouroDireto.csv'

    # Download the stock data
    stock_data = yf.download(tickers, progress=False)
    adj_close = stock_data['Adj Close']
    
    # Function to get the first business day of each month
    def get_first_business_day_of_month(data):
        return data.resample('BMS').first()
    
    monthly_adj_close = adj_close.apply(get_first_business_day_of_month)
    pct_change_df = monthly_adj_close.pct_change() * 100
    pct_change_df.dropna(how='all', inplace=True)
    pct_change_df.reset_index(inplace=True)
    pct_change_df.rename(columns={'Date': 'Data'}, inplace=True)
    
    return pct_change_df

In [2]:
import pandas          as pd
import numpy           as np
import numpy_financial as npf
import requests

import sys
import os

from functools              import reduce
from datetime               import datetime
from dateutil.relativedelta import relativedelta

pd.options.display.float_format = '{:.2f}'.format

In [3]:
headers = {
    'Authorization' : f'Bearer mZjaDnx-pOpLaL__gGOY9jYVzD9HgdLf',
    'Content-Type' : 'application/json'}

# Dá fetch em todas as páginas até retornar um conjunto vazio 
def fetch_all_pages(endpoint, headers):
    all_data = []
    page = 0
    
    while True:
        response = requests.get(f'https://app.bidpatrimonial.com.br/items/{endpoint}?page={page}', headers=headers)
        data = response.json().get('data', [])
        
        if not data:  
            break
        
        all_data.extend(data)
        page += 1
    
    return all_data

imoveis = pd.json_normalize(fetch_all_pages('imovel', headers))[['id_imovel', 'data_compra', 'valor_compra', 'valor_bid_venda']]
finan = pd.json_normalize(fetch_all_pages('financeiro', headers))[['id_transacao','data','categoria','valor','id_imovel.key']]
contratos = pd.json_normalize(fetch_all_pages('contrato', headers))[['id_contrato','data_inicio','valor_aluguel']]

# Define Funções

### Corrige o valor do aluguel para a data de hoje pelo IGPM

In [4]:
# A correção por IGPM funciona da seguinte forma:

# Determinação das Datas de Início e Fim:
# É considerada a data de início do contrato de aluguel e o valor mensal acordado para o mesmo, assim como as variações percentuais mensais do IGPM.
# A data de fim é geralmente a data de hoje ou a data em que se deseja calcular a correção.

# Extração da Série Histórica:
# Extraio a série histórica das variações percentuais do IGPM, considerando o período desde a data de início até a data de hoje.
# Esses dados são obtidos de uma fonte confiável que fornece as variações mensais do IGPM ao longo do tempo.

# Cálculo dos Fatores de Correção:
# Convertemos as variações percentuais mensais do IGPM em fatores de correção. Por exemplo, uma variação de 5% é convertida em um fator de 1.05.
# Calculamos o produto acumulado desses fatores de correção para todo o período. Este produto acumulado representa o fator total de correção.

# Aplicação do Fator de Correção:
# O valor acordado de aluguel é multiplicado pelo fator acumulado de correção para obter o valor corrigido até a data de hoje.
# Este valor corrigido reflete o ajuste necessário para manter o poder de compra do valor do aluguel original, conforme as variações do IGPM ao longo do tempo.

def corrige_aluguel_igpm_hoje(dados_economicos, contratos, data_hoje):
    # Converte a coluna de datas do dataframe de contratos
    contratos['data_inicio'] = pd.to_datetime(contratos['data_inicio'])

    # Função para calcular o IGPM acumulado
    def calcular_igpm_acumulado_aluguel(data_inicio, data_fim, dados_economicos):
        filtro = (dados_economicos['Data'] >= data_inicio) & (dados_economicos['Data'] <= data_fim)
        igpm_periodo = dados_economicos.loc[filtro, 'Igpm']
        igpm_acumulado = (igpm_periodo / 100 + 1).prod() - 1
        return igpm_acumulado

    # Adiciona as novas colunas ao dataframe de contratos
    contratos['valor_aluguel_corrigido'] = contratos.apply(
        lambda row: row['valor_aluguel'] * (1 + calcular_igpm_acumulado_aluguel(row['data_inicio'], data_hoje, dados_economicos)), 
        axis=1)
    contratos['data_aluguel_corrigida'] = data_hoje

    return contratos

### Corrige o valor de compra dos imoveis para a data de hoje por diversos indices

In [5]:
# A correção do valor dos imóveis é feita de forma similar a dos aluguéis por IGPM, o valor inicial é multiplicado pelo fator acumulado ao longo da série histórica

# Nesse caso, a data início de referência é a data de compra do imóvel, e o valor, valor de compra (quando disponíveis).
# O objetivo é fornecer uma informação aproximada do cenário em que aquele capital, naquele período, foi aplicado no investimento ou no índice de referência.

# Lembrando aqui que os investimentos podem possuir diferentes valores de desvio padrão em seu conjunto de retornos (algo que pode ser definido como "risco"), portanto,
# valores futuros podem diferir significativamente de retornos passados. A idéia é dar apenas uma noção aproximada do capital corrigido num cenário alternativo ao investimento no imóvel

def corrige_valores_ativos(data_inicio, data_fim, coluna, df):
    filtro = (df['Data'] >= data_inicio) & (df['Data'] <= data_fim)
    serie = df.loc[filtro, coluna]
    fator_acumulado = (serie / 100 + 1).prod()
    return fator_acumulado

### Calcula rentabilidades totais de cada imovel

In [6]:
# Calculo o rendimento aproximado de cada imóvel.

# O Cálculo é feito da seguinte forma:
# Receita Valorização: O Ganho do proprietário com a valorização do imóvel. É a diferença do valor de compra em relaçao ao valor atual (valor bid, obtido pelo algoritmo de precificação)
# Receita Líquida: O Ganho total do prorietário com o imóvel em todo período, incluindo ganhos com a valorização + rendimentos de aluguel + receitas diversas - gastos gerais - impostos, etc...

# Os valores de operação do imóvel (aluguel, reembolsos, impostos, tarifas, reformas), foram fornecidos em um sheet pelo proprietário, e foram categorizados por mim dessa forma. Os cálculos
# podem mudar na medida em que o proprietário altera informações existentes ou adiciona novas

# Função de cálculo da rentabilidade de imóveis com finan passado como argumento
def calculos_rentabilidade_imoveis(imoveis, finan):
    # Carregando e processando o arquivo finan
    finan = finan[['id_imovel.key', 'categoria', 'valor']]
    finan = finan.query("categoria != 'Movimentacao'")
    
    # Verificar se há valores inválidos na coluna 'valor'
    finan_invalid = finan[~finan['valor'].apply(lambda x: isinstance(x, (int, float)))]
    if not finan_invalid.empty:
        print("Valores inválidos encontrados na coluna 'valor':")
        print(finan_invalid)
        # Remover valores inválidos
        finan = finan.drop(finan_invalid.index)
    
    finan = finan.dropna(subset=['id_imovel.key'])
    finan.columns = ['id_imovel', 'categoria', 'valor']
    
    finan_pivot = finan.pivot_table(index='id_imovel', columns='categoria', values='valor', fill_value=0).reset_index()
    
    # Renomear colunas de acordo com as categorias presentes
    finan_pivot.columns.name = None
    columns_mapping = {col: col for col in finan_pivot.columns}
    if 'Despesa' in finan_pivot.columns:
        columns_mapping['Despesa'] = 'despesa_imovel'
    if 'Receita' in finan_pivot.columns:
        columns_mapping['Receita'] = 'receita_imovel'
    if 'Reembolso' in finan_pivot.columns:
        columns_mapping['Reembolso'] = 'reembolso_imovel'
    
    finan_pivot = finan_pivot.rename(columns=columns_mapping)
    
    # Merge com o DataFrame imoveis
    imoveis = imoveis.merge(finan_pivot, how='left', on='id_imovel')
    
    # Calcular e formatar as colunas receita_valorizacao e receita_liquida
    imoveis['receita_valorizacao'] = (imoveis['valor_bid_venda'] - imoveis['valor_compra'])
    imoveis['receita_liquida'] = (
        imoveis['valor_bid_venda'] - imoveis['valor_compra'] + 
        imoveis.get('despesa_imovel', 0) + imoveis.get('receita_imovel', 0) + imoveis.get('reembolso_imovel', 0))
    
    # Converte data compra para data
    imoveis['data_compra'] = pd.to_datetime(imoveis['data_compra'])
    
    return imoveis

### Calcula a TIR dos imóveis

In [7]:
# Calcula a TIR (Taxa interna de retorno) para cada imóvel, simulando um cenário de rendimento mensal/anual, desconsiderando o risco.

# Função para calcular a TIR mensal
def calcular_tir_mensal(valor_compra, valor_final, periodo):
    if pd.isna(valor_compra) or pd.isna(valor_final) or pd.isna(periodo) or periodo <= 0:
        return np.nan
    fluxos_de_caixa = [-valor_compra] + [0] * (int(periodo) - 1) + [valor_final]
    try:
        tir_mensal = npf.irr(fluxos_de_caixa)
        return tir_mensal
    except (ValueError, np.linalg.LinAlgError):
        return np.nan

# Função para calcular a TIR anual a partir da TIR mensal
def calcular_tir_anual(tir_mensal):
    if pd.isna(tir_mensal):
        return np.nan
    tir_anual = (1 + tir_mensal) ** 12 - 1
    return tir_anual

### Importa Dados

In [8]:
# Define a data de hoje
data_hoje = pd.to_datetime('today').normalize()

# Define os dados economicos
dados_economicos = DadosEconomicos().concatenar_tudo()

# Define dados de ativos financeiros
tickers = ['GC=F', 'IFIX.SA', '^BVSP']
ativos = pega_ativos_yfinance(tickers)

c:\Users\guici\AppData\Local\Programs\Python\Python312\Lib\site-packages\bs4\__init__.py:228: UserWarning: You provided Unicode markup but also provided a value for from_encoding. Your from_encoding will be ignored.
  warnings.warn("You provided Unicode markup but also provided a value for from_encoding. Your from_encoding will be ignored.")
c:\Users\guici\AppData\Local\Programs\Python\Python312\Lib\site-packages\bs4\__init__.py:228: UserWarning: You provided Unicode markup but also provided a value for from_encoding. Your from_encoding will be ignored.
  warnings.warn("You provided Unicode markup but also provided a value for from_encoding. Your from_encoding will be ignored.")
c:\Users\guici\AppData\Local\Programs\Python\Python312\Lib\site-packages\bs4\__init__.py:228: UserWarning: You provided Unicode markup but also provided a value for from_encoding. Your from_encoding will be ignored.
  warnings.warn("You provided Unicode markup but also provided a value for from_encoding. Your f

### Aplica a função para a collection contrato, retorna valores de aluguéis corrigidos

In [9]:
# Supondo que dados_economicos e contratos já estejam definidos e que data_hoje seja a data atual
contratos_corrigidos = corrige_aluguel_igpm_hoje(dados_economicos, contratos, data_hoje)[['id_contrato','valor_aluguel_corrigido','data_aluguel_corrigida']]
# Contratos corrigidos está pronto para ser upado no directus

### Aplica a função para a collections investimentos, retorna nova tabela com dados de cada imovel

In [10]:
# Aplicar a correção para cada índice de dados_economicos
indices_economicos = ['Igpm', 'Ipca', 'Dolar', 'Incc']
colunas_corrigidas_economicos = ['igpm_corr', 'ipca_corr', 'dolar_corr', 'incc_corr']

for indice, coluna_corrigida in zip(indices_economicos, colunas_corrigidas_economicos):
    imoveis[coluna_corrigida] = imoveis.apply(
        lambda row: row['valor_compra'] * corrige_valores_ativos(row['data_compra'], data_hoje, indice, dados_economicos) 
        if pd.notnull(row['data_compra']) else row['valor_compra'], axis=1)

# Aplicar a correção para cada índice de ativos
indices_ativos = ['GC=F', 'IFIX.SA', '^BVSP']
colunas_corrigidas_ativos = ['gc_corr', 'ifix_corr', 'bvsp_corr']

# Aplicar a correção para cada índice de ativos
for indice, coluna_corrigida in zip(indices_ativos, colunas_corrigidas_ativos):
    imoveis[coluna_corrigida] = imoveis.apply(
        lambda row: row['valor_compra'] * corrige_valores_ativos(row['data_compra'], data_hoje, indice, ativos) 
        if pd.notnull(row['data_compra']) else row['valor_compra'], axis=1)

# Adicionar a coluna data_corrigida com a data de hoje
imoveis['data_corrigida'] = data_hoje

# Chamar a função calculos_rentabilidade_imoveis
invests = calculos_rentabilidade_imoveis(imoveis, finan)

# Adicionando novamente a coluna data_corrigida se necessário
invests['data_corrigida'] = data_hoje

# Define o período (número de meses decorridos da compra do imóvel) e valor_final 
# (o valor presente do investimento no imóvel, ou seja, seu valor de compra + toda a receita obtida)
invests['periodo'] = (data_hoje - invests['data_compra']).dt.days / 30.42
invests['periodo'] = invests['periodo'].replace([np.inf, -np.inf, np.nan], 0)
invests['valor_final'] = (invests['valor_compra'] + invests['receita_liquida'])

# Aplicando as funções para calcular a rentabilidade mensal e anual
invests['rent_mensal_est'] = invests.apply(lambda row: calcular_tir_mensal(row['valor_compra'], row['valor_final'], row['periodo']), axis=1)
invests['rent_anual_est'] = invests['rent_mensal_est'].apply(calcular_tir_anual)

# Convertendo as colunas de rentabilidade para strings formatadas com 5 casas decimais
invests['rent_mensal_est'] = invests['rent_mensal_est'].apply(lambda x: f"{x:.5f}" if not pd.isna(x) else x)
invests['rent_anual_est'] = invests['rent_anual_est'].apply(lambda x: f"{x:.5f}" if not pd.isna(x) else x)

# Dropa a coluna movimentacaop
invests.drop(['data_compra','valor_compra','valor_bid_venda','Movimentação'], axis=1, inplace=True)